# Stage 6 — YOLOv8n vs LMD_YOLO (Tesla T4)
Ejecute primero solamente **PREFLIGHT + SMOKE**. Las celdas de 20 épocas están detrás de un gate manual y no se lanzan al ejecutar todo. TEST no se evalúa.

In [ ]:
# 1. Comprobar GPU asignada
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Seleccione Runtime > Change runtime type > T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
assert 'T4' in torch.cuda.get_device_name(0).upper()

In [ ]:
# 2. Montar Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Localizar y extraer el paquete en almacenamiento rápido temporal
from pathlib import Path
import shutil, zipfile
ZIP_PATH = Path('/content/drive/MyDrive/TP1_LMD/stage6/packages/colab_stage6.zip')
PACKAGE_ROOT = Path('/content/colab_stage6')
RESULTS_ROOT = Path('/content/drive/MyDrive/TP1_LMD/stage6/results')
assert ZIP_PATH.is_file(), ZIP_PATH
if PACKAGE_ROOT.exists():
    shutil.rmtree(PACKAGE_ROOT)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall('/content')
assert (PACKAGE_ROOT / 'README.md').is_file()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print(PACKAGE_ROOT, RESULTS_ROOT)

In [ ]:
# 4. Verificar todos los hashes del paquete antes de ejecutar código del proyecto
import subprocess, sys
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/manifests.py'), 'verify-package'], check=True)

In [ ]:
# 5. Instalar dependencias generales; NO instala ultralytics ni reemplaza torch CUDA
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PACKAGE_ROOT/'requirements-colab.txt')], check=True)

In [ ]:
# 6. PREFLIGHT completo: hashes del dataset, modelos, estructura, forward y backward GPU
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage6.py'), 'preflight', '--results-root', str(RESULTS_ROOT)], check=True)

In [ ]:
# 7. SMOKE: 1 época sobre 2% de TRAIN para cada modelo, batch 16
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage6.py'), 'smoke', '--results-root', str(RESULTS_ROOT)], check=True)

In [ ]:
# 8. Revisar el gate. DETENERSE aquí y leer los resultados/VRAM.
import json
preflight = json.loads((RESULTS_ROOT/'preflight.json').read_text())
gate = json.loads((RESULTS_ROOT/'smoke_gate.json').read_text())
print(json.dumps(preflight, indent=2))
print(json.dumps(gate, indent=2))
assert preflight['status'] == gate['status'] == 'PASS'
print('STOP: preflight + smoke completos. No ejecute entrenamiento largo sin revisar estos datos.')

## Gate manual para las 20 épocas
Esta sección no corre por defecto. Solo después de revisar el smoke, cambie conscientemente la variable a `True`.

In [ ]:
RUN_LONG_TRAINING = False  # cambiar manualmente a True tras revisar el smoke
assert RUN_LONG_TRAINING, 'Gate cerrado: no se lanzarán 20 épocas automáticamente.'

In [ ]:
# 9A. YOLOv8n: 20 épocas desde cero
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage6.py'), 'train', '--model', 'yolov8n', '--results-root', str(RESULTS_ROOT)], check=True)

In [ ]:
# 9B. LMD_YOLO TRAIN multirrama: 20 épocas desde cero
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage6.py'), 'train', '--model', 'lmd_yolo', '--results-root', str(RESULTS_ROOT)], check=True)

In [ ]:
# 10. Comparación y curvas; no usa TEST
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage6.py'), 'compare', '--results-root', str(RESULTS_ROOT)], check=True)